# Imports

In [1]:
import numpy as np
import joblib

from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report

# Load Feature Extractor

In [2]:
base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(227, 227, 3)
)

x = base_model.output
x = GlobalAveragePooling2D()(x)

feature_extractor = Model(inputs=base_model.input, outputs=x)

print("_/ VGG16 (AlexNet-style) feature extractor ready")

_/ VGG16 (AlexNet-style) feature extractor ready


# Data Generators

In [3]:
datagen = ImageDataGenerator(rescale=1./255)

train_generator = datagen.flow_from_directory(
    'dataset/train',
    target_size=(227, 227),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

validation_generator = datagen.flow_from_directory(
    'dataset/validation',
    target_size=(227, 227),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

test_generator = datagen.flow_from_directory(
    'dataset/test',
    target_size=(227, 227),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

Found 1145 images belonging to 6 classes.
Found 267 images belonging to 6 classes.
Found 220 images belonging to 6 classes.


# Feature Extraction Function

In [4]:
def extract_features(generator, model):
    features, labels = [], []

    for i in range(len(generator)):
        x_batch, y_batch = generator[i]
        f_batch = model.predict(x_batch, verbose=0)

        features.append(f_batch)
        labels.append(np.argmax(y_batch, axis=1))

    return np.vstack(features), np.hstack(labels)

# Extract Features

In [5]:
print("Extracting TRAIN features...")
X_train, y_train = extract_features(train_generator, feature_extractor)

print("Extracting VALIDATION features...")
X_val, y_val = extract_features(validation_generator, feature_extractor)

print("Extracting TEST features...")
X_test, y_test = extract_features(test_generator, feature_extractor)

print("Shapes:", X_train.shape, X_val.shape, X_test.shape)

Extracting TRAIN features...
Extracting VALIDATION features...
Extracting TEST features...
Shapes: (1145, 512) (267, 512) (220, 512)


# Normalize Features

In [6]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print("_/ Features normalized")

_/ Features normalized


# PCA 

In [7]:
pca = PCA(n_components=200)

X_train = pca.fit_transform(X_train)
X_val = pca.transform(X_val)
X_test = pca.transform(X_test)

print("_/ PCA applied:", X_train.shape)

_/ PCA applied: (1145, 200)


# SVM with GridSearch ***important***

In [8]:
print("Tuning SVM...")

param_grid = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 0.01, 0.001],
    'kernel': ['rbf']
}

grid = GridSearchCV(
    SVC(probability=True),
    param_grid,
    cv=3,
    verbose=2,
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Params:", grid.best_params_)

svm_model = grid.best_estimator_

print("_/ Best SVM model ready")

Tuning SVM...
Fitting 3 folds for each of 9 candidates, totalling 27 fits
Best Params: {'C': 10, 'gamma': 0.001, 'kernel': 'rbf'}
_/ Best SVM model ready


# Validation Evaluation

In [9]:
y_val_pred = svm_model.predict(X_val)

print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))

Validation Accuracy: 0.8352059925093633


# Test Evaluation

In [10]:
y_test_pred = svm_model.predict(X_test)

print("_/ Final Test Accuracy:", accuracy_score(y_test, y_test_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, y_test_pred))

_/ Final Test Accuracy: 0.8090909090909091

Classification Report:

              precision    recall  f1-score   support

           0       0.95      1.00      0.98        40
           1       0.81      0.75      0.78        40
           2       0.71      0.68      0.69        40
           3       0.74      0.88      0.80        40
           4       0.74      0.65      0.69        40
           5       0.95      1.00      0.98        20

    accuracy                           0.81       220
   macro avg       0.82      0.82      0.82       220
weighted avg       0.81      0.81      0.81       220



# Save Model

In [11]:
joblib.dump(svm_model, "alexnet_svm_model.pkl")
joblib.dump(scaler, "alexnet_svm_scaler.pkl")
joblib.dump(pca, "alexnet_svm_pca.pkl")

print("_/ Model saved")

_/ Model saved


# Single Image Prediction

In [12]:
from tensorflow.keras.preprocessing import image

svm_model = joblib.load("alexnet_svm_model.pkl")
scaler = joblib.load("alexnet_svm_scaler.pkl")
pca = joblib.load("alexnet_svm_pca.pkl")

class_indices = train_generator.class_indices
index_to_class = {v: k for k, v in class_indices.items()}

img_path = "dataset/test/A2-Sitting-down/235.png"
img = image.load_img(img_path, target_size=(227,227))

img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

features = feature_extractor.predict(img_array)
features = scaler.transform(features)
features = pca.transform(features)

pred = svm_model.predict(features)[0]
confidence = np.max(svm_model.predict_proba(features))

print("Predicted Class:", index_to_class[pred])
print("Confidence:", confidence)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 304ms/step
Predicted Class: A2-Sitting-down
Confidence: 0.9987723223219962
